# H(12,4): piloto auditado
Ative a GPU no Colab. Execute as duas células seguintes. O piloto não cobre todo o espaço.
O modelo permanece condicionado à necessidade das cinco leis iniciais; nenhum resultado aqui resolve H(16,4).
Os filtros vêm das funções originais; os sobreviventes finais são verificados por Walsh completo. CUDA ainda não foi testado no computador que preparou este arquivo.

In [ ]:
from pathlib import Path
Path("RODAR_H12_A100_AUDITADO.py").write_text('"""Colab/Linux: python RODAR_H12_A100_AUDITADO.py (piloto de 2^22).\nVarredura completa: acrescente --full. Validacao local: --validate-only.\nConclusoes condicionadas as cinco leis iniciais; nao prova nao existencia global.\nSomente biblioteca padrao Python + nvcc/CUDA no computador de execucao.\n"""\nimport argparse\nfrom collections import Counter\nimport hashlib\nfrom itertools import combinations\nimport json\nfrom pathlib import Path\nimport random\nimport shutil\nimport subprocess\nimport time\n\nLAWS = [[40], [42,26], [39,8,14,19,26],\n        [37,0,2,6,12,15,23,25], [36,1,7,9,18,24,32,34],\n        [3,4,5,21,22,26,27,33,35],\n        [0,3,4,5,7,12,21,22,23,24,25,27,32,33,34,35]]\n\n\ndef nullspace(rows, n):\n    piv = {}\n    for v in rows:\n        while v:\n            p = v.bit_length()-1\n            if p in piv:\n                v ^= piv[p]\n            else:\n                piv[p] = v\n                break\n    basis = []\n    for j in range(n):\n        if j in piv:\n            continue\n        v = 1 << j\n        for p in sorted(piv):\n            if (v & piv[p]).bit_count() & 1:\n                v ^= 1 << p\n        basis.append(v)\n    assert all(not ((v&r).bit_count()&1) for v in basis for r in rows)\n    return basis\n\n\ndef orbits(k):\n    result, seen = [], set()\n    for c in combinations(range(12), k):\n        m = sum(1 << i for i in c)\n        if m in seen:\n            continue\n        orb = {((m << r) | (m >> (12-r))) & 4095 for r in range(12)}\n        seen.update(orb)\n        result.append(sorted(orb))\n    return result\n\n\ndef xor_selected(z, columns):\n    v = 0\n    while z:\n        bit = z & -z\n        v ^= columns[bit.bit_length()-1]\n        z ^= bit\n    return v\n\n\ndef prepare():\n    orb = orbits(4)\n    assert len(orb) == 43\n    basis = nullspace([sum(1 << i for i in row) for row in LAWS], 43)\n    assert len(basis) == 36\n    original = [sum((sum((x&m)==m for m in o)&1) << x for x in range(4096)) for o in orb]\n    truth = [xor_selected(v, original) for v in basis]\n    filters = [(o[0],1) for o in orbits(8)] + [(u,2) for u in (0x3cf,0x5d7,0x6db)] + [(4095,3)]\n    masks = [sum(1 << x for x in range(4096) if (x&u)==x) for u,b in filters]\n    # A bit-b of a sum of linear Boolean forms is quadratic if every\n    # nonzero signature multiplicity is divisible by 2^(b-1).\n    # Check that hypothesis on every subcube before interpolating.\n    signatures = [sum(((t >> x)&1) << j for j,t in enumerate(truth)) for x in range(4096)]\n    for u,b in filters:\n        counts = Counter(signatures[x] for x in range(4096) if (x&u)==x)\n        assert all(s==0 or n % (1 << (b-1)) == 0 for s,n in counts.items())\n    def exact(z):\n        t = xor_selected(z, truth)\n        return sum((((t&m).bit_count() >> b)&1) << e for e,((u,b),m) in enumerate(zip(filters,masks)))\n    pairs = list(combinations(range(36),2))\n    v0 = exact(0)\n    singles = [exact(1 << i) for i in range(36)]\n    doubles = [exact((1 << i)|(1 << j)) for i,j in pairs]\n    eqs = []\n    for e,(u,b) in enumerate(filters):\n        c = (v0 >> e)&1\n        lin = sum((((s >> e)&1)^c) << i for i,s in enumerate(singles))\n        rows = [0]*36\n        for (i,j),s in zip(pairs,doubles):\n            if ((s >> e)&1)^c^((lin >> i)&1)^((lin >> j)&1):\n                rows[i] |= 1 << j\n        if c or lin or any(rows):\n            eqs.append((e,c,lin,rows))\n    assert len(eqs) == 42\n    def evaluate(z):\n        values = 0\n        for k,(e,c,lin,rows) in enumerate(eqs):\n            v = c ^ ((z&lin).bit_count()&1)\n            for i in range(36):\n                if (z >> i)&1:\n                    v ^= (z&rows[i]).bit_count()&1\n            values |= v << k\n        return values\n    rng = random.Random(106)\n    checks = [0, (1<<36)-1] + [1 << i for i in range(36)] + [rng.getrandbits(36) for _ in range(256)]\n    for z in checks:\n        raw = exact(z)\n        expected = sum(((raw >> e)&1) << k for k,(e,*_) in enumerate(eqs))\n        assert evaluate(z) == expected\n    return basis,truth,eqs,checks,[evaluate(z) for z in checks]\n\n\nCUDA = r\'\'\'\n#include <cuda_runtime.h>\n#include <cstdio>\n#include <cstdlib>\n#include <cstdint>\n#include <algorithm>\n#define CK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,"CUDA ERROR line %d: %s\\n",__LINE__,cudaGetErrorString(e));exit(2);} } while(0)\nusing U = unsigned long long;\n__constant__ U L[42], Q[42][36], T[36][64], C;\n__host__ __device__ inline unsigned parity(U x) {\n#ifdef __CUDA_ARCH__\n return __popcll(x)&1;\n#else\n return __builtin_popcountll(x)&1;\n#endif\n}\n__device__ U values(U x) {\n U result=0;\n for(int e=0;e<42;e++) {\n  U q=0, bits=x;\n  while(bits){int i=__ffsll((long long)bits)-1; q^=Q[e][i]&x; bits&=bits-1;}\n  result |= U(parity(q)^parity(L[e]&x)^((C>>e)&1))<<e;\n }\n return result;\n}\n__device__ bool passes(U x) {\n for(int e=0;e<42;e++) {\n  U q=0,bits=x;\n  while(bits){int i=__ffsll((long long)bits)-1;q^=Q[e][i]&x;bits&=bits-1;}\n  if(parity(q)^parity(L[e]&x)^((C>>e)&1))return false;\n }\n return true;\n}\n__global__ void validate(const U* input,U* output,unsigned n){\n unsigned i=blockIdx.x*blockDim.x+threadIdx.x;\n if(i<n)output[i]=values(input[i]);\n}\n__global__ void scan(U start,unsigned n,U* found,unsigned* count,U* raw){\n U i=U(blockIdx.x)*blockDim.x+threadIdx.x;\n if(i>=n)return;\n U x=start+i;\n if(!passes(x))return;\n atomicAdd(raw,1ULL);\n unsigned weight=0;\n for(int w=0;w<64;w++){\n  U value=0,bits=x;\n  while(bits){int j=__ffsll((long long)bits)-1;value^=T[j][w];bits&=bits-1;}\n  weight+=__popcll(value);\n }\n // W(0)=4096-2*weight must equal +/-64.\n if(weight!=2016 && weight!=2080)return;\n unsigned p=atomicAdd(count,1u);\n // capacity is n, so no candidate can be truncated.\n found[p]=x;\n}\n// DATA\nint main(int argc,char**argv){\n U start=argc>1?strtoull(argv[1],nullptr,0):0;\n U amount=argc>2?strtoull(argv[2],nullptr,0):(1ULL<<22);\n unsigned chunk=1u<<20;\n if(start>(1ULL<<36)||amount>(1ULL<<36)-start){fprintf(stderr,"Invalid range\\n");return 3;}\n CK(cudaSetDevice(0));cudaDeviceProp prop;CK(cudaGetDeviceProperties(&prop,0));\n printf("GPU %s; conditional domain of 36 variables\\n",prop.name);fflush(stdout);\n CK(cudaMemcpyToSymbol(L,hL,sizeof(hL)));CK(cudaMemcpyToSymbol(Q,hQ,sizeof(hQ)));\n CK(cudaMemcpyToSymbol(T,hT,sizeof(hT)));CK(cudaMemcpyToSymbol(C,&hC,sizeof(hC)));\n U *di,*dout;CK(cudaMalloc(&di,sizeof(tests)));CK(cudaMalloc(&dout,sizeof(tests)));\n CK(cudaMemcpy(di,tests,sizeof(tests),cudaMemcpyHostToDevice));\n validate<<<(NTEST+255)/256,256>>>(di,dout,NTEST);CK(cudaGetLastError());CK(cudaDeviceSynchronize());\n U actual[NTEST];CK(cudaMemcpy(actual,dout,sizeof(actual),cudaMemcpyDeviceToHost));\n for(int i=0;i<NTEST;i++)if(actual[i]!=expected[i]){fprintf(stderr,"CPU/GPU validation FAILED at %d\\n",i);return 4;}\n CK(cudaFree(di));CK(cudaFree(dout));printf("CPU_GPU_VALIDATION_PASS %d\\n",NTEST);fflush(stdout);\n U *found,*raw;unsigned *count;\n CK(cudaMalloc(&found,sizeof(U)*chunk));CK(cudaMalloc(&raw,sizeof(U)));CK(cudaMalloc(&count,sizeof(unsigned)));\n U *host=new U[chunk];U totalRaw=0,totalW0=0,totalBent=0;\n cudaEvent_t a,b;CK(cudaEventCreate(&a));CK(cudaEventCreate(&b));\n for(U offset=0;offset<amount;){\n  unsigned n=(unsigned)std::min(U(chunk),amount-offset);\n  CK(cudaMemset(raw,0,sizeof(U)));CK(cudaMemset(count,0,sizeof(unsigned)));\n  CK(cudaEventRecord(a));scan<<<(n+255)/256,256>>>(start+offset,n,found,count,raw);\n  CK(cudaGetLastError());CK(cudaDeviceSynchronize());CK(cudaEventRecord(b));CK(cudaEventSynchronize(b));\n  float ms;CK(cudaEventElapsedTime(&ms,a,b));unsigned hc;U hr;\n  CK(cudaMemcpy(&hc,count,sizeof(hc),cudaMemcpyDeviceToHost));CK(cudaMemcpy(&hr,raw,sizeof(hr),cudaMemcpyDeviceToHost));\n  if(hc>n){fprintf(stderr,"Corrupt count\\n");return 5;}\n  CK(cudaMemcpy(host,found,sizeof(U)*hc,cudaMemcpyDeviceToHost));\n  for(unsigned k=0;k<hc;k++){\n   U z=host[k],coeff=0;U word[64]={0};\n   for(int j=0;j<36;j++)if((z>>j)&1){coeff^=basis[j];for(int w=0;w<64;w++)word[w]^=hT[j][w];}\n   int walsh[4096];for(int x=0;x<4096;x++)walsh[x]=((word[x/64]>>(x%64))&1)?-1:1;\n   for(int step=1;step<4096;step*=2)for(int p=0;p<4096;p+=2*step)for(int q=0;q<step;q++){\n    int u=walsh[p+q],v=walsh[p+q+step];walsh[p+q]=u+v;walsh[p+q+step]=u-v;\n   }\n   bool bent=true;for(int x=0;x<4096;x++)if(abs(walsh[x])!=64){bent=false;break;}\n   if(bent){totalBent++;printf("{\\"bent\\":true,\\"z\\":%llu,\\"coefficients_hex\\":\\"%llx\\"}\\n",z,coeff);}\n  }\n  totalRaw+=hr;totalW0+=hc;offset+=n;\n  printf("{\\"end_exclusive\\":%llu,\\"tested\\":%llu,\\"quadratic_survivors\\":%llu,\\"w0_survivors\\":%llu,\\"bent\\":%llu,\\"gpu_batch_ms\\":%.3f}\\n",start+offset,offset,totalRaw,totalW0,totalBent,ms);fflush(stdout);\n }\n printf("{\\"status\\":\\"COMPLETE_CONDITIONAL_RANGE\\",\\"start\\":%llu,\\"count\\":%llu,\\"bent\\":%llu}\\n",start,amount,totalBent);fflush(stdout);\n CK(cudaFree(found));CK(cudaFree(raw));CK(cudaFree(count));CK(cudaEventDestroy(a));CK(cudaEventDestroy(b));delete[]host;\n return 0;\n}\n\'\'\'\n\n\ndef main():\n    ap=argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--full\',action=\'store_true\')\n    ap.add_argument(\'--count\',type=int)\n    ap.add_argument(\'--start\',type=int,default=0)\n    ap.add_argument(\'--validate-only\',action=\'store_true\')\n    args=ap.parse_args()\n    count=args.count if args.count is not None else (1<<36 if args.full else 1<<22)\n    if not 0<=args.start<=1<<36 or not 0<count<=(1<<36)-args.start:\n        ap.error(\'Intervalo invalido; ao retomar, informe --start e --count restantes.\')\n    folder=Path(\'h12_a100_auditado\');folder.mkdir(exist_ok=True)\n    t=time.perf_counter()\n    basis,truth,eqs,checks,expected=prepare()\n    scalar=lambda x:f\'{x}ULL\'\n    array=lambda xs:\'{\'+\',\'.join(scalar(x) for x in xs)+\'}\'\n    data=\'\\n\'.join([\n        \'static U hC=\'+scalar(sum(c << k for k,(e,c,l,r) in enumerate(eqs)))+\';\',\n        \'static U hL[42]=\'+array([l for e,c,l,r in eqs])+\';\',\n        \'static U hQ[42][36]={\'+\',\'.join(array(r) for e,c,l,r in eqs)+\'};\',\n        \'static U hT[36][64]={\'+\',\'.join(array([(t>>(64*w))&((1<<64)-1) for w in range(64)]) for t in truth)+\'};\',\n        \'static U basis[36]=\'+array(basis)+\';\',\n        f\'constexpr int NTEST={len(checks)};\',\n        \'static U tests[NTEST]=\'+array(checks)+\';\',\n        \'static U expected[NTEST]=\'+array(expected)+\';\'])\n    source=CUDA.replace(\'// DATA\',data)\n    cu=folder/\'solver.cu\';cu.write_text(source,encoding=\'utf-8\')\n    metadata={\'scope\':\'Conditional on the five initial laws; no global nonexistence claim.\',\n              \'initial_laws\':LAWS[:5],\'additional_laws\':LAWS[5:],\n              \'basis_36_hex\':[hex(v) for v in basis],\'filters\':47,\'active_equations\':42,\n              \'degree_hypothesis\':\'PASS\',\'interpolation_points\':667,\n              \'dense_and_boundary_checks\':len(checks),\n              \'cuda_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n              \'gpu_executed\':False}\n    (folder/\'model.json\').write_text(json.dumps(metadata,indent=2),encoding=\'utf-8\')\n    print(f\'CPU: grau quadratico certificado; 42 equacoes; {len(checks)} controles PASS; {time.perf_counter()-t:.2f}s.\',flush=True)\n    print(\'ESCOPO: dominio condicionado as cinco leis iniciais. Nenhum teorema global sera anunciado.\',flush=True)\n    if args.validate_only:return\n    if not shutil.which(\'nvcc\'):\n        raise SystemExit(\'nvcc ausente. Ative um ambiente CUDA; validacao CPU e fonte foram salvas.\')\n    subprocess.run([\'nvidia-smi\'],check=True)\n    binary=folder/\'solver\'\n    subprocess.run([\'nvcc\',\'-O3\',\'-std=c++14\',str(cu),\'-o\',str(binary)],check=True)\n    name=f\'run_{args.start}_{count}_{time.time_ns()}.log\'\n    with (folder/name).open(\'w\',encoding=\'utf-8\') as log:\n        began=time.perf_counter()\n        with subprocess.Popen([str(binary.resolve()),str(args.start),str(count)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:\n            try:\n                for line in proc.stdout:\n                    print(line,end=\'\',flush=True);log.write(line);log.flush()\n                rc=proc.wait()\n            except BaseException:\n                proc.terminate()\n                try:\n                    proc.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    proc.kill();proc.wait()\n                raise\n    metadata.update(gpu_executed=True,process_exit_code=rc,start=args.start,count=count,log=name)\n    (folder/\'model.json\').write_text(json.dumps(metadata,indent=2),encoding=\'utf-8\')\n    if rc:raise SystemExit(f\'Execucao FALHOU (codigo {rc}); nenhum resultado de completude e valido.\')\n    print(f\'Tempo total: {time.perf_counter()-began:.2f}s. Log: {folder/name}\',flush=True)\n    print(\'Envie model.json e o log. Um piloto cobre somente o intervalo informado.\',flush=True)\n\n\nif __name__==\'__main__\':\n    main()\n', encoding="utf-8")


In [ ]:
%run RODAR_H12_A100_AUDITADO.py


Se o piloto terminar com CPU_GPU_VALIDATION_PASS e COMPLETE_CONDITIONAL_RANGE, execute a célula opcional abaixo para cobrir 2^36 candidatos. A execução completa repetirá o intervalo do piloto.
Baixe a pasta h12_a100_auditado (incluindo model.json e os logs).
Se interromper, o último end_exclusive registra o fim do último lote concluído; retome com --start ESSE_VALOR --count RESTANTE. Evite perder os logs anteriores.

In [ ]:
# Opcional: remova o # para executar a busca completa.
# %run RODAR_H12_A100_AUDITADO.py --full
